## Mecánica de Sólidos
### Ramón Zúñiga — Instituto de Geociencias, UNAM

## Transformación de Coordenadas y Matrices de Cosenos Directores

En este cuaderno se aprende a:
1. Construir una **Matriz de Cosenos Directores (MCD)** a partir de un eje-ángulo o ángulos de Euler.
2. Convertir entre representaciones de rotación (MCD ↔ ángulos de Euler).
3. Transformar **vectores** y **tensores** entre sistemas de coordenadas.
4. Distinguir entre rotación **activa** (el vector gira) y **pasiva** (los ejes giran).

**Paquete principal:** [`ReferenceFrameRotations.jl`](https://juliaspace.github.io/ReferenceFrameRotations.jl/stable/)

---
## 1. Fundamento Teórico: ¿Qué es una MCD?

Una **Matriz de Cosenos Directores** $\mathbf{R} \in \mathbb{R}^{3\times3}$ describe la orientación
de un sistema de coordenadas $(x', y', z')$ respecto a otro $(x, y, z)$.
Cada elemento $R_{ij}$ es el **coseno del ángulo** entre el eje $i$ del nuevo sistema
y el eje $j$ del original:

$$R_{ij} = \cos(\theta_{ij}) = \hat{e}_i^{\,\prime} \cdot \hat{e}_j$$

### Rotación elemental en 2D

Si el sistema $(x', y')$ se obtiene rotando $(x, y)$ un ángulo $\theta$
en sentido antihorario, los nuevos ejes apuntan en las direcciones:

$$\hat{e}_{x'} = \cos\theta\,\hat{e}_x + \sin\theta\,\hat{e}_y$$
$$\hat{e}_{y'} = -\sin\theta\,\hat{e}_x + \cos\theta\,\hat{e}_y$$

Lo que da la **MCD elemental para rotación alrededor de $z$**:

$$\mathbf{R}_z(\theta) =
\begin{bmatrix}
 \cos\theta &  \sin\theta & 0 \\
-\sin\theta &  \cos\theta & 0 \\
 0           &  0           & 1
\end{bmatrix}$$

Las tres rotaciones elementales en 3D son $\mathbf{R}_x$, $\mathbf{R}_y$ y $\mathbf{R}_z$.
Cualquier orientación se puede expresar como su composición.

### Propiedades Fundamentales de las MCD

| Propiedad | Expresión | Significado |
|---|---|---|
| **Ortogonalidad** | $\mathbf{R}^{-1} = \mathbf{R}^T$ | La inversa es gratis: sólo transponer |
| **Determinante** | $\det(\mathbf{R}) = +1$ | Rotación propia (sin reflexión) |
| **Isometría** | $\|\mathbf{R}\mathbf{v}\| = \|\mathbf{v}\|$ | Conserva longitudes y ángulos |
| **Invariantes tensoriales** | $\text{tr}(\boldsymbol{\sigma}') = \text{tr}(\boldsymbol{\sigma})$ | La traza (y det) se conservan bajo rotación |

**Transformación de un tensor de segundo orden** (p.ej., tensor de esfuerzos):

$$\boldsymbol{\sigma}' = \mathbf{R}\,\boldsymbol{\sigma}\,\mathbf{R}^T$$

Esta ley es consecuencia directa de $\mathbf{R}^{-1} = \mathbf{R}^T$.

---
## 2. Importación de Paquetes y Configuración

#### Ejemplo 1: MCD a partir de eje-ángulo

Usamos `angleaxis_to_dcm(ángulo, eje)` para generar una MCD que rota `π/4` (45°) alrededor del vector `[1,1,1]`.

In [20]:
a = π/4;    # definimos un ángulo de rotación
#v = [sqrt(3)/3;sqrt(3)/3;sqrt(3)/3]    # definimos un vector 
v = [1;1;1] 

3-element Vector{Int64}:
 1
 1
 1

In [21]:
MCD = angleaxis_to_dcm(a,v)      #  esto nos encuentra la matriz de cosenos directores para ese vector  

DCM{Float64}:
  1.0        1.0       -0.414214
 -0.414214   1.0        1.0
  1.0       -0.414214   1.0

In [22]:
# newv = MCD*v   #   probamos la transformación

In [23]:
typeof(MCD)

DCM{Float64}

#### La MCD identidad: ninguna rotación

La matriz identidad $\mathbf{I}$ corresponde a una rotación nula (los dos sistemas coinciden). Sus ángulos de Euler son $(0°, 0°, 0°)$.

In [24]:
dcm=DCM([1 0 0; 0 1 0; 0 0 1])    # usamos como ejemplo esta matriz de cosenos directores

DCM{Int64}:
 1  0  0
 0  1  0
 0  0  1

In [25]:
dcm_to_angle(dcm, :XYZ)    #  lo cual nos da los angulos de rotación con respecto a cada eje

EulerAngles{Float64}:
  R(X) :  0.0 rad  ( 0.0°)
  R(Y) :  0.0 rad  ( 0.0°)
  R(Z) :  0.0 rad  ( 0.0°)

In [26]:
dcm_to_angle(MCD, :XYZ)  # probando con la matriz generada anteriormente

EulerAngles{Float64}:
  R(X) :  0.785398 rad  ( 45.0°)
  R(Y) :  1.5708   rad  ( 90.0°)
  R(Z) :  0.0      rad  ( 0.0°)

---
## 3. Ángulos de Euler

Los **ángulos de Euler** describen cualquier orientación 3D como una secuencia de tres
rotaciones elementales sucesivas. La secuencia `:XYZ` de `ReferenceFrameRotations.jl`
aplica primero $R_x$, luego $R_y$ y finalmente $R_z$:

$$\mathbf{R} = R_z(\phi)\,R_y(\theta)\,R_x(\psi)$$

| Ángulo | Eje | Nombre coloquial |
|:---:|:---:|---|
| $\psi$ | X | *Roll* / alabeo |
| $\theta$ | Y | *Pitch* / cabeceo |
| $\phi$ | Z | *Yaw* / guiñada |

> **Importante:** Las rotaciones finitas **no conmutan**: $R_x R_z \neq R_z R_x$ en general.  
> **Gimbal lock:** Para $\theta = \pm 90°$ se pierde un grado de libertad. En ese caso
> conviene usar cuaterniones en lugar de ángulos de Euler.

In [27]:
dcm = angle_to_dcm(pi/4, pi/4, pi/4, :XYZ) # conversión de ángulos de Euler a MCD

DCM{Float64}:
  0.5        0.853553  0.146447
 -0.5        0.146447  0.853553
  0.707107  -0.5       0.5

In [28]:
dcm_to_angle(dcm, :XYZ)        #.  :XYZ especifica la secuencia de rotación

EulerAngles{Float64}:
  R(X) :  0.785398 rad  ( 45.0°)
  R(Y) :  0.785398 rad  ( 45.0°)
  R(Z) :  0.785398 rad  ( 45.0°)

#### Verificación: $\mathbf{R}^{-1} = \mathbf{R}^T$

Para una MCD válida, la inversa coincide con la transpuesta.
El siguiente cálculo lo confirma: `inv(dcm) ≈ dcm'`

In [29]:
idcm = inv(dcm)  # calculamos la inversa de la matriz de cosenos directores

DCM{Float64}:
 0.5       -0.5        0.707107
 0.853553   0.146447  -0.5
 0.146447   0.853553   0.5

---
## 4. Transformación de Vectores

Dado un vector $\mathbf{v}$ expresado en el sistema original, su representación
en el sistema de coordenadas **rotado** es:

$$\mathbf{v}' = \mathbf{R}\,\mathbf{v}$$

donde $\mathbf{R}$ es la MCD que lleva del sistema original al nuevo.
En `ReferenceFrameRotations.jl`, la MCD construida con `angle_to_dcm`
implementa esta transformación directamente.

In [30]:
vu = [1;0;0]   # definimos un vector unitario

3-element Vector{Int64}:
 1
 0
 0

In [31]:
vuprima = idcm*v # calculamos el vector en el sistema coordenado rotado

3-element SVector{3, Float64} with indices SOneTo(3):
 0.7071067811865476
 0.5000000000000002
 1.5

#### Vamos a graficar el vector original y el rotado

In [32]:
pygui(true)  # para generar la ventana de graficación externa

true

#### Definimos los extremos del vector para usar quiver

In [33]:
x = [0;0;0]; y = [0;0;0]; z = [0;0;0]; u = [1;0;0]; v = [0;0;0]; w = [0;0;0]; 
fig = figure()
xlim([-0.0, 1.5])
ylim([-0.0, 1.5])
zlim([-0.0, 1.5])
quiver(x,y,z, u,v, w, lw =0.5)  # esto para inicializar quiver

/Users/ramon/.julia/conda/3/lib/python3.7/site-packages/matplotlib/pyplot.py:935: UserWarning: Requested projection is different from current axis projection, creating new axis with requested projection.
  return gcf().gca(**kwargs)
/Users/ramon/.julia/conda/3/lib/python3.7/site-packages/matplotlib/figure.py:98: MatplotlibDeprecationWarning: 
Adding an axes using the same arguments as a previous axes currently reuses the earlier instance.  In a future version, a new instance will always be created and returned.  Meanwhile, this warning can be suppressed, and the future behavior ensured, by passing a unique label to each axes instance.
  "Adding an axes using the same arguments as a previous axes "


PyObject <mpl_toolkits.mplot3d.art3d.Line3DCollection object at 0x7fcd1a067a90>

In [34]:
fig = figure()
# grafica los vectores en 3D, los ejes se pueden girar interactivamente.
xlim([-0.0, 1.5])
ylim([-0.0, 1.5])
zlim([-0.0, 1.5])

quiver(x,y,z, u,v, w)



PyObject <mpl_toolkits.mplot3d.art3d.Line3DCollection object at 0x7fcd1ac79668>

#### usando los componentes del vector encontrado anteriormente

In [35]:
u = [vuprima[1];0;0]; v = [vuprima[2];0;0]; w = [vuprima[3];0;0];

In [36]:
fig = figure()

xlim([-0.0, 1.5])
ylim([-0.0, 1.5])
zlim([-0.0, 1.5])

quiver(x,y,z, u,v, w)


PyObject <mpl_toolkits.mplot3d.art3d.Line3DCollection object at 0x7fcd1ade2b38>

---
## 5. Rotación Activa vs. Rotación Pasiva — Distinción Fundamental

Esta diferencia es una de las fuentes de confusión más frecuentes en mecánica.

### Rotación Activa
El sistema de coordenadas **permanece fijo** y el vector **gira en el espacio**:

$$\mathbf{v}_{\text{rotado}} = \mathbf{R}\,\mathbf{v}$$

### Rotación Pasiva (transformación de coordenadas)
El vector **permanece fijo** en el espacio, pero los **ejes cambian**.
El mismo vector ahora tiene otras componentes en el nuevo sistema:

$$\mathbf{v}'_{\text{nuevo sistema}} = \mathbf{R}\,\mathbf{v}_{\text{sistema original}}$$

### Relación entre ambas

Una rotación pasiva de $+\theta$ es equivalente a una rotación activa de $-\theta$.
Por eso:

$$\text{Activa: } \mathbf{v}' = \mathbf{R}\,\mathbf{v} \qquad
  \text{Pasiva (vector en eje nuevo): } \mathbf{v}'_{\text{pasiva}} = \mathbf{R}^T\mathbf{v}$$

### Resumen de lo realizado en las celdas anteriores

| Variable | Significado |
|---|---|
| `dcm` | MCD de transformación (pasiva): lleva del sistema original al rotado |
| `dcm * v` | Componentes de `v` en el sistema **rotado** |
| `inv(dcm) * v` = `dcm' * v` | Rotación **activa** de `v`: mueve el vector, ejes fijos |
| `vuprima` | Vector unitario $\hat{x}$ expresado en el sistema de coordenadas **rotado** |

> **Lo que se hizo:** Se rotaron los **ejes de referencia**, no el vector.
> El vector $\hat{x} = [1,0,0]$ permanece fijo en el espacio;
> `vuprima = inv(dcm) * vu` son sus componentes en el sistema de ejes rotados.
> Comparar las dos gráficas: el vector no cambió, pero los ejes sí.

---
## 6. Visualización Completa: Ejes Originales y Rotados 

La siguiente celda grafica en una sola figura:
- Los **tres ejes unitarios del sistema original** (rojo, verde, azul)
- El **vector transformado** en el sistema rotado (magenta)

Esto facilita ver la diferencia entre rotar el vector y rotar los ejes.

In [37]:
# Importar los paquetes necesarios
using ReferenceFrameRotations
using StaticArrays
using PyPlot
PyPlot.PyCall.pyimport("mpl_toolkits.mplot3d")  # necesario para habilitar projection="3d"

# Definir un ángulo de rotación
a = π/4  

# Definir el vector a rotar
v = [1;1;1] 

# Calcular la Matriz de Cosenos Directores (MCD) para la rotación alrededor del vector v
MCD = angleaxis_to_dcm(a, v)  

# Verificar el tipo de la matriz
println(typeof(MCD))

# Definir una MCD de ejemplo (matriz identidad)
dcm = DCM([1 0 0; 0 1 0; 0 0 1])    

# Convertir la MCD a ángulos de Euler
println("Ángulos de Euler de la matriz identidad:", dcm_to_angle(dcm, :XYZ))

# Convertir la MCD calculada a ángulos de Euler
println("Ángulos de Euler de la MCD calculada:", dcm_to_angle(MCD, :XYZ))

# Convertir ángulos de Euler a MCD
dcm = angle_to_dcm(π/4, π/4, π/4, :XYZ) 

# Obtener los ángulos de Euler de la nueva MCD
println("Ángulos de Euler de la nueva MCD:", dcm_to_angle(dcm, :XYZ))    

# Calcular la inversa de la MCD
idcm = inv(dcm)  

# Definir un vector unitario
vu = [1;1;1]  

# Calcular el vector transformado en el sistema de coordenadas rotado
vuprima = idcm * vu 

# Habilitar gráficos interactivos
pygui(true)

# Punto de origen común
x0 = [0]; y0 = [0]; z0 = [0]

# Ejes rotados expresados en el sistema ORIGINAL
# dcm lleva del sistema rotado al original: v_orig = dcm * v_rot
# → las columnas de dcm son los vectores x', y', z' en el sistema original
dcm_mat = Matrix(dcm)
xp = dcm_mat[:, 1]   # x' en sistema original
yp = dcm_mat[:, 2]   # y' en sistema original
zp = dcm_mat[:, 3]   # z' en sistema original

# ── Figura 1: Sistema original con el vector vu ─────────────────────────────
fig1 = figure()
ax1 = fig1.add_subplot(111, projection="3d")
ax1.set_xlim([-0.2, 1.2]); ax1.set_ylim([-0.2, 1.2]); ax1.set_zlim([-0.2, 1.2])

ax1.quiver(x0, y0, z0, [1], [0], [0], color="r", label="Eje X")
ax1.quiver(x0, y0, z0, [0], [1], [0], color="g", label="Eje Y")
ax1.quiver(x0, y0, z0, [0], [0], [1], color="b", label="Eje Z")
ax1.quiver(x0, y0, z0, [vu[1]], [vu[2]], [vu[3]],
           color="k", linewidth=2, label="vu = $(round.(vu, digits=3))")

ax1.set_xlabel("X"); ax1.set_ylabel("Y"); ax1.set_zlabel("Z")
ax1.legend(); ax1.set_title("Vector vu en el sistema original")
show()

# ── Figura 2: Sistema original, sistema rotado y vector vu ──────────────────
fig2 = figure()
ax2 = fig2.add_subplot(111, projection="3d")
ax2.set_xlim([-1.0, 1.2]); ax2.set_ylim([-1.0, 1.2]); ax2.set_zlim([-0.5, 1.2])

# Ejes del sistema ORIGINAL (colores claros, transparentes)
ax2.quiver(x0, y0, z0, [1], [0], [0], color="r",        alpha=0.25, label="Eje X")
ax2.quiver(x0, y0, z0, [0], [1], [0], color="g",        alpha=0.25, label="Eje Y")
ax2.quiver(x0, y0, z0, [0], [0], [1], color="b",        alpha=0.25, label="Eje Z")

# Ejes del sistema ROTADO (colores sólidos, más gruesos, etiquetas con prima)
ax2.quiver(x0, y0, z0, [xp[1]], [xp[2]], [xp[3]], color="darkred",   linewidth=2, label="Eje X'")
ax2.quiver(x0, y0, z0, [yp[1]], [yp[2]], [yp[3]], color="darkgreen", linewidth=2, label="Eje Y'")
ax2.quiver(x0, y0, z0, [zp[1]], [zp[2]], [zp[3]], color="darkblue",  linewidth=2, label="Eje Z'")

# Vector original vu (negro, grueso)
ax2.quiver(x0, y0, z0, [vu[1]], [vu[2]], [vu[3]],
           color="k", linewidth=3, label="vu = $(round.(vu, digits=3))")

# Componentes de vu en el sistema rotado (magenta, grueso)
ax2.quiver(x0, y0, z0, [vuprima[1]], [vuprima[2]], [vuprima[3]],
           color="m", linewidth=3, label="vu en sist. rotado = $(round.(collect(vuprima), digits=3))")

ax2.set_xlabel("X"); ax2.set_ylabel("Y"); ax2.set_zlabel("Z")
ax2.legend(loc="upper left", fontsize=8)
ax2.set_title("Sistema original (transparente)  |  Sistema rotado (oscuro)  |  vu (negro → magenta)")
show()


DCM{Float64}
Ángulos de Euler de la matriz identidad:EulerAngles{Float64}: R(XYZ)  0.0  0.0  0.0 rad
Ángulos de Euler de la MCD calculada:EulerAngles{Float64}: R(XYZ)  0.785398  1.5708  0.0 rad
Ángulos de Euler de la nueva MCD:EulerAngles{Float64}: R(XYZ)  0.785398  0.785398  0.785398 rad


---
## Ejercicios Propuestos

**Ej. 1 — Rotación de 90° alrededor del eje Z.**  
Construye la MCD para una rotación de $90°$ alrededor de $z$. Aplícala al vector $\mathbf{v} = [1,\,0,\,0]^T$. Verifica el resultado analíticamente.

---
**Ej. 2 — No conmutatividad de rotaciones.**  
Compara la rotación $R_x(45°)\cdot R_z(45°)$ con $R_z(45°)\cdot R_x(45°)$ aplicadas al vector $[1,\,1,\,0]^T$. ¿El orden importa?

---
**Ej. 3 — Recuperación de ángulos de Euler.**  
Genera una MCD con ángulos $(\pi/6,\,\pi/4,\,\pi/3)$ en secuencia XYZ. Recupera los ángulos con `dcm_to_angle`. ¿Obtienes los mismos valores de entrada?

---
**Ej. 4 — Transformación de un tensor de esfuerzos.**  
Un tensor diagonal $\boldsymbol{\sigma} = \text{diag}(10,\,5,\,2)$ MPa (sistema principal) se rota $45°$ alrededor de $z$ con la ley $\boldsymbol{\sigma}' = \mathbf{R}\,\boldsymbol{\sigma}\,\mathbf{R}^T$.  
¿Aparecen esfuerzos cortantes? ¿Se conserva la traza (invariante $I_1$)?

In [38]:
using ReferenceFrameRotations, LinearAlgebra

# ── Ejercicio 1: Rotación 90° alrededor de Z ──────────────────────────────────
println("=== Ejercicio 1: Rotación 90° alrededor de Z ===")
R90z = angle_to_dcm(0.0, 0.0, π/2, :XYZ)
v    = [1.0, 0.0, 0.0]
v_rot = R90z * v
println("Vector original:  ", v)
println("Vector rotado:    ", round.(v_rot, digits=10))
# Resultado analítico: [1,0,0] rotado 90° en Z → [0,-1,0] (o [0,1,0] según conv.)
# TODO: prueba también con v = [0,1,0] y v = [0,0,1]

# ── Ejercicio 2: No conmutatividad de rotaciones ──────────────────────────────
println("\n=== Ejercicio 2: No conmutatividad ===")
Rx = angle_to_dcm(π/4, 0.0, 0.0, :XYZ)
Rz = angle_to_dcm(0.0, 0.0, π/4, :XYZ)
v  = [1.0, 1.0, 0.0]

v_xz = Rx * (Rz * v)   # primero Z, luego X
v_zx = Rz * (Rx * v)   # primero X, luego Z
println("v después de Rz→Rx: ", round.(v_xz, digits=6))
println("v después de Rx→Rz: ", round.(v_zx, digits=6))
println("¿Iguales? ", v_xz ≈ v_zx)
# TODO: ¿hay algún caso especial en que las rotaciones sí conmuten?

# ── Ejercicio 3: Recuperación de ángulos de Euler ─────────────────────────────
println("\n=== Ejercicio 3: Recuperación de ángulos de Euler ===")
θx, θy, θz = π/6, π/4, π/3
dcm = angle_to_dcm(θx, θy, θz, :XYZ)
ea  = dcm_to_angle(dcm, :XYZ)
println("Ángulos originales  (°): ",
    round(rad2deg(θx), digits=2), ", ",
    round(rad2deg(θy), digits=2), ", ",
    round(rad2deg(θz), digits=2))
println("Ángulos recuperados: ", ea)
# TODO: prueba con la secuencia :ZYX. ¿Cambian los ángulos recuperados?

# ── Ejercicio 4: Transformación de tensor de esfuerzos ────────────────────────
println("\n=== Ejercicio 4: Transformación de tensor σ ===")
σ_p = [10.0 0.0 0.0; 0.0 5.0 0.0; 0.0 0.0 2.0]   # MPa, ejes principales
R   = Matrix(angle_to_dcm(0.0, 0.0, π/4, :XYZ))    # rotación 45° en Z
σ_r = R * σ_p * R'
println("σ en ejes principales (MPa):"); display(σ_p)
println("σ rotado 45° en Z (MPa):");    display(round.(σ_r, digits=6))
println("¿Aparecen cortantes? ", !isapprox(σ_r, Diagonal(diag(σ_r)), atol=1e-8))
println("Traza original: ", tr(σ_p), " MPa  |  Traza rotada: ", round(tr(σ_r), digits=10), " MPa")
# TODO: verifica que det(σ) también se conserva bajo rotación

=== Ejercicio 1: Rotación 90° alrededor de Z ===
Vector original:  [1.0, 0.0, 0.0]
Vector rotado:    [0.0, -1.0, 0.0]

=== Ejercicio 2: No conmutatividad ===
v después de Rz→Rx: [1.414214, 0.0, -0.0]
v después de Rx→Rz: [1.207107, -0.207107, -0.707107]
¿Iguales? false

=== Ejercicio 3: Recuperación de ángulos de Euler ===
Ángulos originales  (°): 30.0, 45.0, 60.0
Ángulos recuperados: EulerAngles{Float64}: R(XYZ)  0.523599  0.785398  1.0472 rad

=== Ejercicio 4: Transformación de tensor σ ===
σ en ejes principales (MPa):


3×3 Matrix{Float64}:
 10.0  0.0  0.0
  0.0  5.0  0.0
  0.0  0.0  2.0

σ rotado 45° en Z (MPa):


3×3 Matrix{Float64}:
  7.5  -2.5  0.0
 -2.5   7.5  0.0
  0.0   0.0  2.0

¿Aparecen cortantes? true
Traza original: 17.0 MPa  |  Traza rotada: 17.0 MPa
